# MiniProyecto Deep Learning
## Elaborado por Fabián Calvo Castillo - Florencia Pellegrini

In [ ]:
#%pip install feedparser trafilatura
#%pip install google-generativeai

In [ ]:
#%pip install python-dotenv

In [ ]:
#%pip install edge-tts
#%pip install nest_asyncio
#%pip install openai-whisper

In [ ]:
#%pip install moviepy==1.0.3
#%pip install Pillow

In [1]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)
from news_fetcher import get_all_news
import google.generativeai as genai
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin
import t2s
importlib.reload(t2s)
from t2s import generar_audio

C:\Users\fcalv\AppData\Local\Temp\ipykernel_22736\2596570605.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Paso 1: Obtener noticias

In [2]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)

from news_fetcher import get_all_news
articulos = get_all_news(num_per_source=3)

RECOPILANDO NOTICIAS
\n-> Leyendo RSS de todas las fuentes...
  [ABC] 0 articulos en RSS
  [ElDiario] 3 articulos en RSS
  [Europa Press] 3 articulos en RSS
  [La Vanguardia] 3 articulos en RSS
  inutos] 3 articulos en RSS
  [Antena 3] 3 articulos en RSS
  [RTVE] 3 articulos en RSS
  [El Pais] 3 articulos en RSS
  [El Mundo] 3 articulos en RSS
\n-> Extrayendo texto completo (24 articulos)...
  Procesando: Sánchez alerta de que los efectos de Irán serán "mucho ...
    -> Texto completo (10910 chars)
  Procesando: La Diputación de Valencia ficha en comisión de servicio...
    -> Texto completo (3015 chars)
  Procesando: La fundación financiada por Vox declaró 630.000 euros e...
    -> Texto completo (6212 chars)
  Procesando: Guerra Irán | Directo: Sánchez acusa a PP y Vox de ayud...
    -> Texto completo (806 chars)
  Procesando: Feijóo llama a Sánchez "pacifista de pacotilla" por usa...
    -> Texto completo (6269 chars)
  Procesando: Irán advierte a EEUU que "no volverá a ver" los pre

Con esta función se guarda una variable articulos, que es una lista de diccionarios Python, uno por cada artículo. 
Así luego el LLM recibirá todos los textos completos agrupados por tema y devolverá un único resumen redactado de forma neutral, sin subjetivismo político. 

In [3]:
# Comprobación

for i, art in enumerate(articulos):
    print(f"\n{'='*40}")
    print(f"Artículo {i+1}")
    print(f"Fuente:  {art['fuente']}")
    print(f"Título:  {art['titulo']}")
    print(f"Origen:  {art['texto_origen']}")
    print(f"Chars:   {len(art['texto_completo'])}")
    print(f"Preview: {art['texto_completo'][:200]}...")


Artículo 1
Fuente:  ElDiario
Título:  Sánchez alerta de que los efectos de Irán serán "mucho peores" que los de Irak y defiende su 'no a la guerra' frente a Aznar
Origen:  completo
Chars:   10910
Preview: Sánchez alerta de que los efectos de Irán serán “mucho peores” que los de Irak y defiende su 'no a la guerra' frente a Aznar
5
Pedro Sánchez alerta de que el mundo que amanecerá tras la guerra de Esta...

Artículo 2
Fuente:  ElDiario
Título:  La Diputación de Valencia ficha en comisión de servicios a la pareja de Pérez Llorca por el doble de su salario anterior
Origen:  completo
Chars:   3015
Preview: La Diputación de Valencia ficha en comisión de servicios a la pareja de Pérez Llorca por el doble de su salario anterior
LEER ESTE TEXTO EN CATALÁN
La Diputación de Valencia ha fichado en comisión de ...

Artículo 3
Fuente:  ElDiario
Título:  La fundación financiada por Vox declaró 630.000 euros en pagos por "servicios a profesionales independientes"
Origen:  completo
Chars:   6212
Prev

## Paso 2: Generación del resumen

Se toma el diccionario generado con las noticias que se recopilaron en distintos medios, y se genera el resumen haciendo uso de la API de Gemini

In [ ]:
import google.generativeai as genai
import importlib
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin

#load_dotenv()

#GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GEMINI_API_KEY = ""

genai.configure(api_key=GEMINI_API_KEY)
mi_modelo_gemini = genai.GenerativeModel("gemini-2.5-flash")

# Pipeline de resumen
boletin_final, resumenes = resumir_noticias(articulos, mi_modelo_gemini)

if boletin_final:
    print("\n" + "*"*50)
    print("BOLETÍN DE NOTICIAS")
    print("*"*50 + "\n")
    print(boletin_final)
    guardar_boletin(boletin_final) 


RESUMIENDO 24 ARTÍCULOS

-> Agrupando artículos por temas...
   3 temas identificados:
   - Debate político español sobre la guerra y el conflicto de Irán (10 fuentes: ElDiario, Europa Press, Europa Press, La Vanguardia, La Vanguardia, 20minutos, 20minutos, 20minutos, El Pais, El Pais)
   - Conflicto de Irán y política exterior (3 fuentes: Europa Press, RTVE, El Pais)
   - Política regional andaluza (2 fuentes: El Mundo, El Mundo)

-> Generando resúmenes por tema...
   [1/3] Debate político español sobre la guerra y el conflicto de Irán (10 fuentes)...
      OK (925 chars)
      Error: type object 'datetime.time' has no attribute 'sleep'
   [2/3] Conflicto de Irán y política exterior (3 fuentes)...
      OK (813 chars)
      Error: type object 'datetime.time' has no attribute 'sleep'
   [3/3] Política regional andaluza (2 fuentes)...
      OK (730 chars)
      Error: type object 'datetime.time' has no attribute 'sleep'

-> Montando boletín final...

Boletín generado: 3263 chars, 3 tem

## Paso 3: Síntesis de voz (T2S)

In [5]:
import importlib
import t2s
importlib.reload(t2s)
from t2s import generar_audio

ruta_audio, ruta_subtitulos = generar_audio(boletin_final)

Generando audio...
  Voz:         es-ES-AlvaroNeural
  Chars:       3263
  Audio:       audios/audio_2026-03-25_11-31-38.mp3
Audio generado! (1.13 MB)
  Generando subtítulos con Whisper...


c:\Users\fcalv\anaconda3\envs\MCD_25_26\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  Subtítulos generados: audios/subtitulos_2026-03-25_11-31-38.vtt


In [6]:
# Verificar que el VTT tiene contenido
with open(ruta_subtitulos, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"VTT tiene {len(contenido)} chars")
print(contenido[:300])

VTT tiene 7847 chars
WEBVTT

00:00:00.000 --> 00:00:01.199
Muy buenas tardes y

00:00:01.199 --> 00:00:02.480
bienvenidos a nuestro boletín

00:00:02.480 --> 00:00:03.140
informativo.

00:00:04.179 --> 00:00:05.580
Hoy analizamos las últimas

00:00:05.580 --> 00:00:07.099
novedades en política nacional

00:00:07.099 -->


## Paso 4: Generación del video resumen 

In [7]:
import subprocess
resultado = subprocess.run(["where", "magick"], capture_output=True, text=True)
print(resultado.stdout)

C:\Program Files\ImageMagick-7.1.2-Q16-HDRI\magick.exe



In [ ]:
import importlib
import video_maker
importlib.reload(video_maker)
from video_maker import generar_video

#PEXELS_API_KEY = os.getenv('PEXELS_API_KEY')
PEXELS_API_KEY = ""

ruta_video = generar_video(
    ruta_audio, ruta_subtitulos, resumenes,
    PEXELS_API_KEY,
    modelo_ia=mi_modelo_gemini  # para mejorar las queries de imagen
)


GENERANDO VÍDEO

1. Cargando audio...
   Duración total:    197.3s
   Temas:             3
   Duración por tema: 65.8s

2. Buscando imágenes en Pexels...
  Tema: Debate político español sobre la guerra y el confl...
    Query imagen: 'Parliamentary Crisis'
    Keywords: 'parliamentary crisis'
    Imagen guardada: imagenes/img_730177792881343214.jpg
  Tema: Conflicto de Irán y política exterior...
    Query imagen: 'Tehran Standoff'
    Keywords: 'tehran standoff'
    Imagen guardada: imagenes/img_1996693011673855832.jpg
  Tema: Política regional andaluza...
    Query imagen: 'Andalusian Parliament'
    Keywords: 'andalusian parliament'
    Imagen guardada: imagenes/img_9170231642975884680.jpg

3. Creando clips...
   [1/3] Debate político español sobre la guerra ...
   [2/3] Conflicto de Irán y política exterior...
   [3/3] Política regional andaluza...

4. Concatenando clips y añadiendo audio...

5. Exportando vídeo base...

5b. Escalando a resolución completa...

6. Añadiendo subtítu